# Private multi-layout invoice dataset
This notebook creates OCR drafts and a source-verified training set. Select a **T4 GPU** runtime. Copy the Windows PDF folder to a private Google Drive folder first. PDFs, labels, and rendered pages stay in your Drive and are never added to the public repository.

In [ ]:
#@title 1. Mount Drive and configure private paths
from google.colab import drive
drive.mount('/content/drive')
PDF_DIR = '/content/drive/MyDrive/sample PDF OCR' #@param {type:'string'}
WORK_DIR = '/content/drive/MyDrive/invoice_ocr_private_training' #@param {type:'string'}
VERIFIED_BY = '' #@param {type:'string'}
import pathlib
if not pathlib.Path(PDF_DIR).is_dir():
    raise ValueError('PDF_DIR is not available in Colab. Upload/copy the folder to Google Drive and update PDF_DIR.')
print('Private PDF folder:', PDF_DIR)
print('Private work folder:', WORK_DIR)

In [ ]:
#@title 2. Download the current public code (no invoice data)
import io, os, pathlib, shutil, urllib.request, zipfile
PROJECT_DIR = pathlib.Path('/content/OCR')
request = urllib.request.Request('https://api.github.com/repos/ubaid-148/OCR/zipball/main', headers={'Accept':'application/vnd.github+json','User-Agent':'OCR-Private-Dataset'})
with urllib.request.urlopen(request, timeout=120) as response:
    archive_bytes = response.read()
extract_root = pathlib.Path('/content/ocr-download')
shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(PROJECT_DIR, ignore_errors=True)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(extract_root)
shutil.move(str(next(path for path in extract_root.iterdir() if path.is_dir())), str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print('Code ready at', PROJECT_DIR)

In [ ]:
#@title 3. Install isolated GPU OCR environment
import os, pathlib, shutil, subprocess, sys
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi','-L'], capture_output=True).returncode == 0
if not gpu_runtime:
    raise RuntimeError('GPU runtime required: Runtime > Change runtime type > T4 GPU')
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable,'-m','venv','--without-pip',str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR/'bin'/'python')
ocr_pip = [sys.executable,'-m','pip','--python',OCR_PYTHON]
subprocess.run([*ocr_pip,'install','-q','--upgrade','pip'], check=True)
subprocess.run([*ocr_pip,'install','-q','torch==2.9.1+cpu','--index-url','https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([*ocr_pip,'install','-q','paddleocr==3.7.0','pypdfium2==5.13.0'], check=True)
subprocess.run([*ocr_pip,'uninstall','-y','paddlepaddle','paddlepaddle-gpu'], check=True)
subprocess.run([*ocr_pip,'install','-q','paddlepaddle-gpu==3.3.1','-i','https://www.paddlepaddle.org.cn/packages/stable/cu126/'], check=True)
OCR_ENV = os.environ.copy()
OCR_ENV.update(OCR_DEVICE='gpu:0', OCR_TARGETED_RETRY='true', USE_LOCAL_AI='false', FLAGS_use_mkldnn='0')
subprocess.run([OCR_PYTHON,'-u',str(PROJECT_DIR/'check_ocr_runtime.py')], cwd=PROJECT_DIR, env=OCR_ENV, check=True)
print('Private drafting runtime ready.')

In [ ]:
#@title 4. Render pages and initialize resumable annotations
import subprocess
subprocess.run([OCR_PYTHON,'-m','training.invoice_dataset','prepare','--pdf-dir',PDF_DIR,'--work-dir',WORK_DIR,'--dpi','200'], cwd=PROJECT_DIR, env=OCR_ENV, check=True)

In [ ]:
#@title 5. Generate OCR drafts for all PDFs (resumable; may take several minutes)
subprocess.run([OCR_PYTHON,'-u','-m','training.invoice_dataset','draft','--work-dir',WORK_DIR,'--languages','ara+eng'], cwd=PROJECT_DIR, env=OCR_ENV, check=True)
print('Drafts are not ground truth. Verify them below against every source page.')

In [ ]:
#@title 6. Review and verify labels against the PDFs
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from training.colab_annotator import launch
dashboard = launch(WORK_DIR, VERIFIED_BY)

In [ ]:
#@title 7. Validate annotation progress
from training.invoice_dataset import validation_report
import json
report = validation_report(pathlib.Path(WORK_DIR))
print(json.dumps(report, ensure_ascii=False, indent=2))
if report['errors']:
    raise ValueError('Fix annotation errors before export.')

In [ ]:
#@title 8. Export supplier/layout-isolated train, validation and test splits
MIN_VERIFIED = 80 #@param {type:'integer'}
from training.invoice_dataset import export_qwen
summary = export_qwen(pathlib.Path(WORK_DIR), min_verified=MIN_VERIFIED)
print(json.dumps(summary, indent=2))
print('Private SFT files:', pathlib.Path(WORK_DIR)/'exports')

When export succeeds, start a **fresh GPU runtime** and open `colab_train.ipynb`. Do not move the private workspace into `/content/OCR` and do not commit it to GitHub.